Data Validation and Cross - Reference

In [0]:
orders_stage="eccom_catalog.default.orders_stage"
customers_stage="eccom_catalog.default.customers_stage"
products_stage="eccom_catalog.default.product_stage"
inventory_stage="eccom_catalog.default.inventory_stage"
shipping_stage="eccom_catalog.default.shipping_stage"
validation_table="eccom_catalog.default.validation_results_table"
print("Starting comprehensive data validation process...")

In [0]:
from pyspark.sql.functions import col,hour,current_timestamp
from datetime import datetime
try:
	#Read all stage table
	orders_stage=spark.read.table(orders_stage)
	customers_stage=spark.read.table(customers_stage)
	products_stage=spark.read.table(products_stage)
	inventory_stage=spark.read.table(inventory_stage)
	shipping_stage=spark.read.table(shipping_stage)
	total_order_records=orders_stage.count()
	total_customer_records=customers_stage.count()
	total_product_records=products_stage.count()
	total_inventory_records=inventory_stage.count()
	total_shipping_records=shipping_stage.count()

	print(f"Total orders records: {total_order_records}")
	print(f"Total customers records: {total_customer_records}")
	print(f"Total product records: {total_product_records}")
	print(f"Total inventory records: {total_inventory_records}")
	print(f"Total shipping records: {total_shipping_records}")

except Exception as e:
	print(f"Failed to read stage tables: {str(e)}")
	raise

In [0]:
#Cross - reference validation: Orders vs Customers 
try:
	#Check for orphaned orders (orders without valid customers)
	orders_without_customer=orders_stage.join(customers_stage,on="customer_id",how="left_anti")
	total_orders_without_customer=orders_without_customer.count()
	
	#Check for orphaned customers (customers without any orders)
	customer_without_orders=customers_stage.join(orders_stage,on="customer_id",how="left_anti")
	total_customer_without_orders=customer_without_orders.count()
	print(f"Total orders without customer: {total_orders_without_customer}")
	print(f"Total customer without orders: {total_customer_without_orders}")
	
	#Validate orders amount are unreasonable
	unreasonable_orders=orders_stage.filter((col("order_amount")<1) | (col("order_amount")>10000))
	total_unreasonable_orders=unreasonable_orders.count()
	print(f"Orders with unreasonable amount: {total_unreasonable_orders}")

except Exception as e:
	print(f"Error in orders-customers validation: {str(e)}")
	raise

In [0]:
#Cross - reference validation: Orders vs Product
try:
	#Check for orphaned orders (orders without valid products)
	orders_without_valid_products=orders_stage.join(products_stage,on="product_id",how="left_anti")
	total_orders_without_valid_products=orders_without_valid_products.count()
	
	#Check for orphaned products (products without any orders)
	product_without_orders=products_stage.join(orders_stage,on="product_id",how="left_anti")
	total_product_without_orders=product_without_orders.count()
	print(f"Total orders without valid products: {total_orders_without_valid_products}")
	print(f"Total product without orders: {total_product_without_orders}")
	
	#Validate order amounts against product prices
	price_mismatch=orders_stage.join(products_stage,on="product_id",how="inner").filter((col("order_amount")-col("price"))>0.01)
	total_price_mismatch=price_mismatch.count()
	print(f"Orders with price - mismatch: {total_price_mismatch}")

except Exception as e:
	print(f"Error in orders-product validation: {str(e)}")
	raise

In [0]:
#Cross - reference validation: Orders vs Shipping
try:
	#Check for orders without shipping informations
	orders_without_shipping=orders_stage.join(shipping_stage,on="order_id",how="left_anti")
	total_orders_without_shipping=orders_without_shipping.count()
	
	#Check for shipping without orders
	shipping_without_orders=shipping_stage.join(orders_stage,on="order_id",how="left_anti")
	total_shipping_without_orders=shipping_without_orders.count()
	print(f"Total orders without shipping: {total_orders_without_shipping}")
	print(f"Total shipping without orders: {total_shipping_without_orders}")
	
	#Validate shipping costs are unreasonable
	unreasonable_shipping=shipping_stage.filter((col("shipping_cost")<0) | (col("shipping_cost")>100))
	total_unreasonable_shipping=unreasonable_shipping.count()
	print(f"Orders with unreasonable shipping: {total_unreasonable_shipping}")

except Exception as e:
	print(f"Error in orders-shipping validation: {str(e)}")
	raise

In [0]:
#Cross - reference validation Products vs Inventory 
try:
	#Check for products without inventory 
	products_without_inventory=products_stage.join(inventory_stage,on="product_id",how="left_anti")
	total_products_without_inventory=products_without_inventory.count()
	
	#Check for inventory without products 
	inventory_without_products=inventory_stage.join(products_stage,on="product_id",how="left_anti")	
	total_inventory_without_products=inventory_without_products.count()
	print(f"Total products without inventory: {total_products_without_inventory}")
	print(f"Total inventroy without products: {total_inventory_without_products}")
	
	#Validate stock quantities are consistent
	products_with_inventory=products_stage.join(inventory_stage,on="product_id",how="inner")
	stock_mismatch=products_with_inventory.filter(products_stage.stock_quantity!=inventory_stage.stock_quantity) 
	total_stock_mismatch=stock_mismatch.count()
	print(f"Products with stock - mismatch: {total_stock_mismatch}")

except Exception as e:
	print(f"Error in products-inventory validation: {str(e)}")
	raise 

In [0]:
#Businnes Rules Validation
try:
	#Rule 1: Premium customers should have higher order values
	premium_customers=customers_stage.join(orders_stage,on="customer_id",how="inner").filter(col("customer_tier")=="premium")
	low_value_premium_orders=premium_customers.filter(col("order_amount")<100)
	total_low_value_premium_orders=low_value_premium_orders.count()

	#Rule 2: Orders should be processed within business hours (8am-6pm)
	orders_outside_hours=orders_stage.filter((hour(col("created_timestamp"))<8) | (hour(col("created_timestamp"))>18))
	total_order_outside_hours=orders_outside_hours.count()

	#Rule 3: Discontinued products should not have new orders
	discontinued_orders=orders_stage.join(products_stage, on="product_id", how="inner").filter((col("discontinued").cast("boolean"))==True)
	total_discontinued_orders=discontinued_orders.count()

	print(f"Total low value premium orders: {total_low_value_premium_orders}")
	print(f"Total orders outside hours: {total_order_outside_hours}")
	print(f"Total orders which have discontinued products: {total_discontinued_orders}")

except Exception as e:
	print(f"Error in business rules validation: {str(e)}")
	raise

In [0]:
#Compile validation results
from pyspark.sql.functions import lit
try:
	Validation_results=[
	{
	"validation_type": "orphaned_orders",
	"count": total_orders_without_customer,
	"severity": "High" if total_orders_without_customer > 0 else "None",
	"description":"Orders without valid customers"
	},
	{
	"validation_type": "orphaned_customers",
	"count": total_customer_without_orders,
	"severity": "High" if total_customer_without_orders > 0 else "None",
	"description":"Customers without any orders"
	},
	{
	"validation_type": "unreasonable_orders",
	"count": total_unreasonable_orders,
	"severity": "High" if total_unreasonable_orders > 0 else "None",
	"description":"Orders with unreasonable amounts"
	},
	{
	"validation_type": "orphaned_orders_products",
	"count": total_orders_without_valid_products,
	"severity": "High" if total_orders_without_valid_products > 0 else "None",
	"description":"Orders with invalid products"
	},
	{
	"validation_type": "product_without_orders",
	"count": total_orders_without_valid_products,
	"severity": "High" if total_orders_without_valid_products > 0 else "None",
	"description":"Products without any orders"
	},
	{
	"validation_type": "price_mismatch",
	"count": total_price_mismatch,
	"severity": "High" if total_price_mismatch > 0 else "None",
	"description":"Orders with price mismatches"
	},
	{
	"validation_type": "orders_without_shipping",
	"count": total_orders_without_shipping,
	"severity": "High" if total_orders_without_shipping > 0 else "None",
	"description":"Orders without shipping information"
	},
	{
	"validation_type": "shipping_without_orders",
	"count": total_shipping_without_orders,
	"severity": "High" if total_shipping_without_orders > 0 else "None",
	"description":"Shipping with invalid orders"
	},
	{
	"validation_type": "unreasonable_shipping",
	"count": total_unreasonable_shipping,
	"severity": "High" if total_shipping_without_orders > 0 else "None",
	"description":"Shipping with unreasonable costs"
	},
	{
	"validation_type": "products_without_inventory",
	"count": total_products_without_inventory,
	"severity": "High" if total_shipping_without_orders > 0 else "None",
	"description":"Products without inventory"
	},
	{
	"validation_type": "inventory_without_products",
	"count": total_inventory_without_products,
	"severity": "High" if total_inventory_without_products > 0 else "None",
	"description":"Inventory without products"
	},
	{
	"validation_type": "products_stock_mismatch_with_inventory",
	"count": total_stock_mismatch,
	"severity": "High" if total_stock_mismatch > 0 else "None",
	"description":"Product stock quantities are inconsistent with inventory"
	},
	{
	"validation_type": "low_value_premium_orders",
	"count": total_low_value_premium_orders,
	"severity": "LOW" if total_low_value_premium_orders > 0 else "NONE",
	"description": "Premium customers with low-value orders"
	},
	{
	"validation_type": "orders_outside_hours",
	"count": total_order_outside_hours,
	"severity": "LOW" if total_order_outside_hours > 0 else "NONE",
	"description": "Orders outside business hours"
	},
	{
	"validation_type": "discontinued_orders",
	"count": total_discontinued_orders,
	"severity": "HIGH" if total_discontinued_orders > 0 else "NONE",
	"description": "Orders for discontinued products"
	}
	]	

	#Create Dataframe from Validation_results
	validation_data=spark.createDataFrame(Validation_results)
	validation_data=validation_data.withColumn("validation_timestamp", current_timestamp())\
	.withColumn("batch_id", lit(datetime.now().strftime("%Y%m%d_%H%M%S")))
	validation_data.write.format("delta").mode("append").saveAsTable(validation_table)

	#Calculate overall validation score
	high_severity_issues=sum(1 for result in Validation_results if result["severity"]=="HIGH")
	medium_severity_issues=sum(1 for result in Validation_results if result["severity"]=="Medium")
	low_severity_issues=sum(1 for result in Validation_results if result["severity"]=="Low")
	overall_status="PASS" if high_severity_issues==0 else "FAIL"   
	print(f"Validation Summary: ")
	print(f"High Severity issues: {high_severity_issues}")
	print(f"Medium Severity issues: {medium_severity_issues}")
	print(f"Overall status: {overall_status}")

except Exception as e:
	print(f"Error compiling validation results: {str(e)}")
	raise

In [0]:
#Log validation Summary
from pyspark.sql.types import StructType, StructField, LongType, StringType
import json
validation_summary={
"archived_files": None,
"invalid_records": None,
"status": None,
"task": "data_validation",
"timestamp": datetime.now().isoformat(),
"total_records": None,
"valid_records": None
}
print(f"Validation Summary: {json.dumps(validation_summary, indent=2)}")

# Explicit schema for processing_log table
log_schema=StructType([
StructField("archived_files", LongType(), True),
StructField("invalid_records", LongType(), True),
StructField("status", StringType(), True),
StructField("task", StringType(), True),
StructField("timestamp", StringType(), True),
StructField("total_records", LongType(), True),
StructField("valid_records", LongType(), True)
])
validation_log=spark.createDataFrame([validation_summary], schema=log_schema)
validation_log.write.format("delta").mode("append").saveAsTable("eccom_catalog.default.processing_log")

#Set validation result for downstream tasks
try:
	dbutils.jobs.taskValues.set("validation_status", overall_status)
	dbutils.jobs.taskValues.set("high_severity_count", high_severity_issues)
	print(f"Task values set successfully - Status: {overall_status}, High severity: {high_severity_issues}")
except Exception as e:
	print(f"Warning: Could not set task values (non-critical): {e}")
	print(f"Validation completed with status: {overall_status}")